In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import numpy as np
import os
from datetime import date
import warnings

warnings.filterwarnings("ignore")

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [2]:

today_date = date.today()
today_date=''.join(str(today_date).split('-'))

project_name='LAGUARDIA'

file_name='LAGUARDIA_KINGElvis_no_aa.csv'
detail_df=pd.read_excel("details_laguardia-terminal_od_excel_template.xlsx",sheet_name='STOPS')
df=pd.read_csv('elvislaguardia2024intercept_export_odbc_filled.csv')
# elvis_df=pd.read_excel(file_name,sheet_name='Elvis_Review')
elvis_df=pd.read_csv(file_name)


In [3]:
if file_name.split('_')[0].isdigit():
    file_first_name=file_name.split('_')[0]+'_'+file_name.split('_')[1]
else:
    file_first_name=file_name.split('_')[0]

elvis_date_check=['elvisdate']
elvis_date=check_all_characters_present(elvis_df,elvis_date_check)

df = df.merge(elvis_df[[elvis_date[0], 'id', 'Final_Usage','FINAL_REVIEWER']], on='id', how='left')

df=df[df['Final_Usage'].str.lower()=='use']

home_airport_hotel_column_names=['originaddresslat','originaddresslong', 'destinaddresslat',
                                 'destinaddresslong','originplacetype','homeaddresslat','homeaddresslong',
                                 'destinairportcode','destinplacetype']


elvis_status_column_check=['elvisstatus']
elvis_status_column=check_all_characters_present(df,elvis_status_column_check)

df=df[df['ELVIS_STATUS'].str.lower()!='delete']


In [5]:
df=df[df['id']==635]

In [6]:
df

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS,Elvis_Date,Final_Usage,FINAL_REVIEWER
42,635,2024-07-28 09:02:58,87.0,en,2024-07-28 08:55:40,2024-07-28 09:02:58,172.58.229.176,https://laguardia-ny.etc-research.com/index.ph...,-14400,STR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-08-19,Use,


In [7]:

blank_columns_checks=['originaddresslat', 'originaddresslong', 'destinaddresslat',
                      'destinaddresslong','stoponlat', 'stoponlong', 'stopofflat', 'stopofflong']
blank_column_names=check_all_characters_present(df,blank_columns_checks)



df.dropna(subset=blank_column_names, how='any',inplace=True)


In [8]:

boarding_columns_checks=['prevtran1onbuslat', 'prevtran1onbuslong',
                         'prevtran2onbuslat', 'prevtran2onbuslong',
                         'prevtran3onbuslat', 'prevtran3onbuslong', 
                         'prevtran4onbuslat', 
                         'prevtran4onbuslong','stoponlat', 'stoponlong',
                         'stopofflat', 'stopofflong',
                          'nexttran1offbuslat','nexttran1offbuslong',  
                         'nexttran2offbuslat', 'nexttran2offbuslong', 
                          'nexttran3offbuslat', 'nexttran3offbuslong', 
                          'nexttran4offbuslat', 'nexttran4offbuslong',]
boarding_columns=check_all_characters_present(df,boarding_columns_checks)
boarding_columns.sort()

origin_destin_columns_checks=['originaddresslat','originaddresslong', 'destinaddresslat', 'destinaddresslong']
origin_destin_columns=check_all_characters_present(df,origin_destin_columns_checks)
origin_destin_columns.sort()


df['FIRST_BOARDING_LAT']=None 
df['FIRST_BOARDING_LONG']=None
df['LAST_ALIGHTING_LAT']=None
df['LAST_ALIGHTING_LONG']=None
df['ORIGIN_TO_SURVEYBOARD']=None    #FINAL_ORIGIN_LOCATION TO STOP_ON_LOCATION (IN MILES)
df['ORIGIN_TO_FIRST_BOARD']=None    #FINAL_ORIGIN_LOCATION TO FIRST_BOARDING_LOCATION (IN MILES)
df['SURVEYBOARDING_TO_SURVEYALIGHTING']=None    #STOP_ON_LOCATION TO STOP_OFF_LOCATION (SURVEYED ROUTE) (IN MILES)
df['ORIGIN_TO_DESTINATION']=None    #FINAL_ORIGIN_LOCATION TO FINAL_DESTIN_LOCATION (IN MILES)
df['SURVEYALIGHTING_TO_DESTINATION']=None    #STOP_OFF_LOCATION TO FINAL_DESTIN LOCATION (IN MILES)
df['LAST_ALIGHTING_LOCATION_TO_DESTIN']=None   #LAST_ALIGHTING_LOCATION TO FINAL_DESTIN LOCATION (IN MILES)


In [9]:

def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        print(f"Error calculating distance: {e}")  # Change to the desired distance unit


In [10]:
boarding_columns

['NEXT_TRAN_1_OFF_BUS_LAT',
 'NEXT_TRAN_1_OFF_BUS_LONG',
 'NEXT_TRAN_2_OFF_BUS_LAT',
 'NEXT_TRAN_2_OFF_BUS_LONG',
 'NEXT_TRAN_3_OFF_BUS_LAT',
 'NEXT_TRAN_3_OFF_BUS_LONG',
 'NEXT_TRAN_4_OFF_BUS_LAT',
 'NEXT_TRAN_4_OFF_BUS_LONG',
 'PREV_TRAN_1_ON_BUS_LAT',
 'PREV_TRAN_1_ON_BUS_LONG',
 'PREV_TRAN_2_ON_BUS_LAT',
 'PREV_TRAN_2_ON_BUS_LONG',
 'PREV_TRAN_3_ON_BUS_LAT',
 'PREV_TRAN_3_ON_BUS_LONG',
 'PREV_TRAN_4_ON_BUS_LAT',
 'PREV_TRAN_4_ON_BUS_LONG',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG']

In [11]:
boarding_columns[8],boarding_columns[9]

('PREV_TRAN_1_ON_BUS_LAT', 'PREV_TRAN_1_ON_BUS_LONG')

In [12]:
boarding_columns[10],boarding_columns[11]

('PREV_TRAN_2_ON_BUS_LAT', 'PREV_TRAN_2_ON_BUS_LONG')

In [13]:
boarding_columns[12],boarding_columns[13]

('PREV_TRAN_3_ON_BUS_LAT', 'PREV_TRAN_3_ON_BUS_LONG')

In [14]:
boarding_columns[14],boarding_columns[15]

('PREV_TRAN_4_ON_BUS_LAT', 'PREV_TRAN_4_ON_BUS_LONG')

In [15]:
boarding_columns[18],boarding_columns[19]

('STOP_ON_LAT', 'STOP_ON_LONG')

In [16]:
for index, row in df.iterrows():
    if not pd.isna(row[boarding_columns[8]]) and not pd.isna(row[boarding_columns[9]]):
        #'PREV_TRAN_1_ON_BUS_LAT',
        #'PREV_TRAN_1_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[8]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[9]]
    elif not pd.isna(row[boarding_columns[10]]) and not pd.isna(row[boarding_columns[11]]):
        #  'PREV_TRAN_2_ON_BUS_LAT',
        # 'PREV_TRAN_2_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[10]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[11]]
    elif not pd.isna(row[boarding_columns[12]]) and not pd.isna(row[boarding_columns[13]]):
        #  'PREV_TRAN_3_ON_BUS_LAT',
        # 'PREV_TRAN_3_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[12]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[13]]
    elif not pd.isna(row[boarding_columns[14]]) and not pd.isna(row[boarding_columns[15]]):
        #  'PREV_TRAN_4_ON_BUS_LAT',
        # 'PREV_TRAN_4_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[14]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[15]]
    elif not pd.isna(row[boarding_columns[18]]) and not pd.isna(row[boarding_columns[19]]):
        #  'STOP_ON_LAT',
        # 'STOP_ON_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[18]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[19]]
    else:
        df.loc[index, 'FIRST_BOARDING_LAT'] = np.nan
        df.loc[index, 'FIRST_BOARDING_LONG'] = np.nan
    #      
    if not pd.isna(row[boarding_columns[6]]) and not pd.isna(row[boarding_columns[7]]):
        #  'NEXT_TRAN_4_OFF_BUS_LAT',
        # 'NEXT_TRAN_4_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[6]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[7]]
    elif not pd.isna(row[boarding_columns[4]]) and not pd.isna(row[boarding_columns[5]]):
        #  'NEXT_TRAN_3_OFF_BUS_LAT',
        # 'NEXT_TRAN_3_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[4]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[5]]
    elif not pd.isna(row[boarding_columns[2]]) and not pd.isna(row[boarding_columns[3]]):
        #  'NEXT_TRAN_2_OFF_BUS_LAT',
        # 'NEXT_TRAN_2_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[2]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[3]]
    elif not pd.isna(row[boarding_columns[0]]) and not pd.isna(row[boarding_columns[1]]):
        #  'NEXT_TRAN_1_OFF_BUS_LAT',
        # 'NEXT_TRAN_1_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[0]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[1]]
    elif not pd.isna(row[boarding_columns[16]]) and not pd.isna(row[boarding_columns[17]]):
        #  'STOP_OFF_LAT',
        # 'STOP_OFF_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[16]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[17]]
    else:
        df.loc[index, 'LAST_ALIGHTING_LAT'] = np.nan
        df.loc[index, 'LAST_ALIGHTING_LONG'] = np.nan


In [17]:
df[['FIRST_BOARDING_LAT','FIRST_BOARDING_LONG','LAST_ALIGHTING_LAT','LAST_ALIGHTING_LONG']]

,FIRST_BOARDING_LAT,FIRST_BOARDING_LONG,LAST_ALIGHTING_LAT,LAST_ALIGHTING_LONG
42,40.772924,-73.872198,40.807957,-73.96422


In [19]:
df[[boarding_columns[18],boarding_columns[19],boarding_columns[16],boarding_columns[17]]]

,STOP_ON_LAT,STOP_ON_LONG,STOP_OFF_LAT,STOP_OFF_LONG
42,40.772924,-73.872198,40.807957,-73.96422


In [20]:
for index, row in df.iterrows():
    df.loc[index,'ORIGIN_TO_SURVEYBOARD']=get_distance_between_coordinates(row[origin_destin_columns[2]],row[origin_destin_columns[3]], row[boarding_columns[18]],row[boarding_columns[19]])
    df.loc[index,'ORIGIN_TO_FIRST_BOARD']=get_distance_between_coordinates(row[origin_destin_columns[2]],row[origin_destin_columns[3]],row['FIRST_BOARDING_LAT'],row['FIRST_BOARDING_LONG'])
    df.loc[index,'SURVEYBOARDING_TO_SURVEYALIGHTING']=get_distance_between_coordinates(row[boarding_columns[18]],row[boarding_columns[19]],row[boarding_columns[16]],row[boarding_columns[17]])
    df.loc[index,'ORIGIN_TO_DESTINATION']=get_distance_between_coordinates(row[origin_destin_columns[2]],row[origin_destin_columns[3]],row[origin_destin_columns[0]],row[origin_destin_columns[1]])
    df.loc[index,'SURVEYALIGHTING_TO_DESTINATION']=get_distance_between_coordinates(row[boarding_columns[16]],row[boarding_columns[17]],row[origin_destin_columns[0]],row[origin_destin_columns[1]])
    df.loc[index,'LAST_ALIGHTING_LOCATION_TO_DESTIN']=get_distance_between_coordinates(row['LAST_ALIGHTING_LAT'],row['LAST_ALIGHTING_LONG'],row[origin_destin_columns[0]],row[origin_destin_columns[1]])

In [21]:
df[['ORIGIN_TO_SURVEYBOARD','ORIGIN_TO_FIRST_BOARD','SURVEYBOARDING_TO_SURVEYALIGHTING','ORIGIN_TO_DESTINATION','SURVEYALIGHTING_TO_DESTINATION','LAST_ALIGHTING_LOCATION_TO_DESTIN']]

,ORIGIN_TO_SURVEYBOARD,ORIGIN_TO_FIRST_BOARD,SURVEYBOARDING_TO_SURVEYALIGHTING,ORIGIN_TO_DESTINATION,SURVEYALIGHTING_TO_DESTINATION,LAST_ALIGHTING_LOCATION_TO_DESTIN
42,0.0,0.0,5.397659,5.307403,0.091146,0.091146


In [22]:
df['O2B/O2D']=None   #df['ORIGIN_TO_SURVEYBOARD']/df['ORIGIN_TO_DESTINATION'] ORIGIN_TO_BOARD Divide by ORIGIN_TO_DESTINATION
df['B2A/OD']=None   #df['SURVEYBOARDING_TO_SURVEYALIGHTING']/df['ORIGIN_TO_DESTINATION'] BOARDING_TO_ALIGHTING Divide by ORIGIN_TO_DESTINATION
df['A2D/OD']=None   #df['SURVEYALIGHTING_TO_DESTINATION']/df['ORIGIN_TO_DESTINATION'] ALIGHTING_TO_DESTINATION Divide by ORIGIN_TO_DESTINATION

for index, row in df.iterrows():
    origin_to_destination = row['ORIGIN_TO_DESTINATION']
    if origin_to_destination==0:
        df.loc[index,'O2B/O2D']=0
        df.loc[index,'B2A/OD']=0
        df.loc[index,'A2D/OD']=0
    else:
        df.loc[index,'O2B/O2D']=row['ORIGIN_TO_SURVEYBOARD']/row['ORIGIN_TO_DESTINATION']
        df.loc[index,'B2A/OD']=row['SURVEYBOARDING_TO_SURVEYALIGHTING']/row['ORIGIN_TO_DESTINATION']
        df.loc[index,'A2D/OD']=row['SURVEYALIGHTING_TO_DESTINATION']/row['ORIGIN_TO_DESTINATION']

In [23]:
df[['O2B/O2D','B2A/OD','A2D/OD']]

,O2B/O2D,B2A/OD,A2D/OD
42,0.0,1.017006,0.017173


In [24]:
transport_transfer_columns_checks=['origintransport','destintransport','nexttransferscode','prevtransferscode']
transport_transfer_columns=check_all_characters_present(df,transport_transfer_columns_checks)
transport_transfer_columns.sort()
transport_transfer_columns

['DESTIN_TRANSPORT',
 'NEXT_TRANSFERSCode',
 'ORIGIN_TRANSPORT',
 'PREV_TRANSFERSCode']

In [27]:
df[[transport_transfer_columns[1],transport_transfer_columns[3]]]

,NEXT_TRANSFERSCode,PREV_TRANSFERSCode
42,0.0,NaN


In [29]:
df[[transport_transfer_columns[1], transport_transfer_columns[3]]]=df[[transport_transfer_columns[1], transport_transfer_columns[3]]].fillna(0)

In [30]:
df[[transport_transfer_columns[1], transport_transfer_columns[3]]]

,NEXT_TRANSFERSCode,PREV_TRANSFERSCode
42,0.0,0.0


In [31]:

walk=['walk','wheelchair or scooter','other','walked','skateboard','bike, e-bike, skateboard, scooter, e-scooter','wheelchair','walked or used mobility aid']
drive=['was dropped off by someone','drove alone and parked','drove or rode with others and parked','taxi','uber, lyft, etc.',
       'get in a parked vehicle & drive alone','be picked up by someone','taxi / shuttle','get in a parked vehicle & drive, alone or w/others',
       'get in a parked vehicle & drive/ride w/others','get in a parked vehicle & drive, alone or w/others','rode with others and was dropped off',
      'rode in an uber / lyft / taxi / etc. vehicle','get in a parked vehicle & drive alone'
      ]


o_b_check1 = (df['ORIGIN_TO_FIRST_BOARD'] > 1.85) & df[transport_transfer_columns[2]].str.lower().isin(walk)
o_b_check2 = (df['ORIGIN_TO_FIRST_BOARD'] < 0.25) & df[transport_transfer_columns[2]].str.lower().isin(drive)
o_b_check3 = (df['ORIGIN_TO_SURVEYBOARD'] < 0.25) & (df[transport_transfer_columns[3]] != 0)

In [32]:
df['O_B_Dist_Check1'] = np.where(o_b_check1, 1, 0)
df['O_B_Dist_Check2'] = np.where(o_b_check2, 1, 0)
df['O_B_Dist_Check3'] = np.where(o_b_check3, 1, 0)

In [33]:
df[['O_B_Dist_Check3']]

,O_B_Dist_Check3
42,0
